<a href="https://colab.research.google.com/github/FBL9207-source/Healthcare-ETL-MiniProject/blob/main/Mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 2**


1.   The data source for this project is a Healthcare Dataset provided in CSV format (healthcare_dataset.csv). The dataset contains records of patients, their medical conditions, hospital admissions, treatments, billing information, and other healthcare-related details.

2.   I chose this dataset because it contains a variety of real-world healthcare information that can be analyzed using Python, Pandas, data visualization, and statistical techniques. It provides an opportunity to understand patient demographics, medical conditions, hospital admissions, billing amounts, medications, and test results. This makes it suitable for performing data cleaning, exploratory data analysis, and extracting useful insights from healthcare data.


3. The dataset contains 55,500 patient records and 15 columns. The main information includes:

    Patient details: Name, Age, Gender, and Blood Type
    Medical details: Medical Condition, Medication, and Test Results
    Hospital information: Doctor, Hospital, Room Number, and Admission Type
    Admission details: Date of Admission and Discharge Date
    Financial information: Billing Amount
    Insurance information: Insurance Provider

    Overall, the dataset provides information about patients and their healthcare journeys, from hospital admission to treatment and discharge.

4.  Data Type: Structured data

5.  Data Format: CSV (Comma-Separated Values)

6.  Data Update Frequency: Static dataset; it is not regularly updated and represents a fixed collection of healthcare records.



**Step 3**

         
            Healthcare Dataset
                     │
                     ▼
                  Extract
                     │
                     ▼
               Validate Data
                     │
                     ▼
                 Clean Data
                     │
                     ▼
              Transform Data
                     │
                     ▼
             Analyze Data
                     │
                     ▼
              Generate Reports
                     │
                     ▼
                   Load

**Step 4**

Healthcare_ETL_Project/
│  
├── raw_data/   
├── reports/    
├── output/         
└── Mini_project.ipynb

In [2]:
import pandas as pd
import os

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Extract

def extract():
  input_path = ("/content/drive/MyDrive/data set/healthcare_dataset.csv")
  data = pd.read_csv(input_path)
  print("Number of records:", data.shape[0])
  print("Number of columns:", data.shape[1])

  os.makedirs("/content/Healthcare_ETL_Project/raw_data", exist_ok=True) # Create raw_data folder
  data.to_csv("/content/Healthcare_ETL_Project/raw_data/healthcare_dataset.csv", index=False)
  return data

In [8]:
# Step 6 –

def transform(data):

  # 1. Remove duplicate records
  data.drop_duplicates(inplace=True)

  # 2.Rename columns (lowercase, replace spaces with underscores)
  data.columns = data.columns.str.lower().str.replace(' ', '_')

  # 3.Standardize name columns values
  data['name'] = data['name'].str.strip().str.title()

  # 4.Sort records by admission date
  data = data.sort_values(by='date_of_admission').reset_index(drop=True)

  # 5. Convert date formats (Date_of_Admission and Discharge_Date)
  data['date_of_admission'] = pd.to_datetime(data['date_of_admission'])
  data['discharge_date'] = pd.to_datetime(data['discharge_date'])

  # 6. Create calculated column: Length of Stay (in days)
  data['length_of_stay'] = (data['discharge_date'] - data['date_of_admission']).dt.days
  # Handle cases where discharge date is before admission date or missing dates result in NaT
  data.loc[data['length_of_stay'] < 0, 'length_of_stay'] = 0 # Assume 0 for invalid stays


  # 7. Standardize 'Gender' values
  data['gender'] = data['gender'].replace({'Male': 'M', 'Female': 'F', 'Other': 'O'})


  # 8. Filter records - Remove records with potentially invalid 'Billing Amount' (e.g., negative)
  initial_billing_rows = data.shape[0]
  data = data[data['billing_amount'] >= 0]
  # 7. Standardize 'Gender' values
  data['gender'] = data['gender'].replace({'Male': 'M', 'Female': 'F', 'Other': 'O'})


  # 8. Filter records - Remove records with potentially invalid 'Billing Amount' (e.g., negative)
  initial_billing_rows = data.shape[0]
  data = data[data['billing_amount'] >= 0]
  print(f"\nRemoved {initial_billing_rows - data.shape[0]} records with negative 'billing_amount'. New shape: {data.shape}")

  print(f"\nFinal data shape after transformations: {data.shape}")
  print(data)
  return data


**Step 7**

In [39]:
def generate_reports(data):
    os.makedirs("/content/Healthcare_ETL_Project/reports", exist_ok=True)

    # 1. Complete cleaned dataset
    data.to_csv(os.path.join("/content/Healthcare_ETL_Project/reports/cleaned_dataset.csv"), index=False)

    # 2. Top 10 records (highest billing amount)
    top_10 = data.sort_values("billing_amount", ascending=False).head(10)
    top_10.to_csv(os.path.join("/content/Healthcare_ETL_Project/reports/highest_billing.csv"), index=False)

    # 3. Bottom 10 records (lowest billing amount)
    bottom_10 = data.sort_values("billing_amount", ascending=True).head(10)
    bottom_10.to_csv(os.path.join("/content/Healthcare_ETL_Project/reports/lowest_billing.csv"), index=False)

    # 4. Summary statistics (numeric columns)
    summary_stats = data.describe()
    summary_stats.to_csv("/content/Healthcare_ETL_Project/reports/summary_statistics.csv")

    print("Reports saved to reports/")

**Step 8**

In [43]:
def load(data):
  os.makedirs("/content/Healthcare_ETL_Project/output", exist_ok=True)

  output_path = "/content/Healthcare_ETL_Project/output/final_cleaned_healthcare_data.csv"
  data.to_csv(output_path, index=False)

  print(f"Final dataset loaded to: {output_path}")
  print(f"Final shape: {data.shape}")

  # Confirm reports are in place
  reports_dir = "/content/Healthcare_ETL_Project/reports"
  report_files = os.listdir(reports_dir)
  print(f"\n{len(report_files)} report(s) available in reports/:")
  for f in sorted(report_files):
    print(f"  - {f}")

**Step 9**

In [47]:
def run_pipeline():
  raw = extract()
  transformed = transform(raw)
  generate_reports(transformed)
  load(transformed)

if __name__ == "__main__":
  run_pipeline()

Number of records: 55500
Number of columns: 15

Removed 0 records with negative 'billing_amount'. New shape: (54860, 16)

Final data shape after transformations: (54860, 16)
                       name  age gender blood_type medical_condition  \
0           Nicole Matthews   27      M         A-           Obesity   
1         Danielle Arellano   38      F         O-           Obesity   
2      Timothy Shepherd Phd   54      F         B+            Cancer   
3            Adam Hernandez   46      F         B-      Hypertension   
4              Angela Green   68      M         A-         Arthritis   
...                     ...  ...    ...        ...               ...   
54961        Sandra English   82      M         O+         Arthritis   
54962           Ralph Moore   52      F         B+           Obesity   
54963            Wendy Hill   74      F        AB+            Asthma   
54964           Sherri Wolf   54      F         A-         Arthritis   
54965  Christopher Clements   54  